<a href="https://colab.research.google.com/github/pejmanrasti/Big_Data/blob/main/04_pyspark_movie_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark MovieLens


## 0. Setup (Run this cell)

In [ ]:
!pip install -q pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Student_Exercises_Movies")
    .master("local[*]")
    .getOrCreate()
)

spark

## 1. Download & Load MovieLens Dataset

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

ratings = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
movies  = spark.read.csv("ml-latest-small/movies.csv", header=True, inferSchema=True)

## Task 1 — Data Exploration

**1.1 Inspect the data**

*	Show first 10 rows of ratings
*	Print schema for both DataFrames
*	Count the number of rows in each




**1.2 Column selection**

Write PySpark code to:

* select only userId, movieId, rating
*	rename movieId → film_id
*	cast rating to integer or float explicitly

**1.3 Filtering**

Using ratings:

*	Find all ratings from userId = 1
*	Find all ratings greater than 4.5
*	Find all ratings on movieId in (1, 50, 100)


(use isin())

**1.4 Sorting & limiting**

*	Show the top 20 highest ratings
*	Show the 10 lowest ratings made by user 600
*	Sort by rating desc and timestamp asc

**1.5 Derived columns**

Create columns:

*	rating_x2 = rating * 2
*	positive_rating = 1 if rating ≥ 4 else 0
*	log_rating = log10(rating + 1)

**1.6 Missing values**

Even if ML-latest-small has no nulls, :

*	Add a fake null column
*	Demonstrate fill/replace/drop operations

**1.7 Distinct & deduplication**

*	Count unique users
*	Count unique movies rated
*	Drop duplicate ratings where (userId, movieId) might repeat (even though dataset is clean)

In [ ]:
print("Ratings DataFrame - First 10 rows:")
ratings.show(10)
print("\nMovies DataFrame - First 10 rows:")
movies.show(10)
print("\nRatings DataFrame Schema:")
ratings.printSchema()
print("\nMovies DataFrame Schema:")
movies.printSchema()
print(f"\nNumber of rows in ratings DataFrame: {ratings.count()}")
print(f"Number of rows in movies DataFrame: {movies.count()}")


# 1.2 Column selection
from pyspark.sql.functions import col
# Select, rename, and cast columns
ratings_selected = ratings.select(col("userId"), col("movieId").alias("film_id"), col("rating").cast("float"))
print("\nRatings DataFrame after column selection, renaming, and casting:")
ratings_selected.show(5)
ratings_selected.printSchema()

# 1.3 Filtering
print("\nRatings from userId = 1:")
ratings.filter(ratings.userId == 1).show(5)
print("\nRatings greater than 4.5:")
ratings.filter(ratings.rating > 4.5).show(5)
print("\nRatings on movieId in (1, 50, 100):")
ratings.filter(ratings.movieId.isin([1, 50, 100])).show(5)

# 1.4 Sorting & limiting
print("\nTop 20 highest ratings:")
ratings.orderBy(col("rating").desc()).show(20)
print("\n10 lowest ratings made by user 600:")
ratings.filter(ratings.userId == 600).orderBy(col("rating").asc()).show(10)
print("\nRatings sorted by rating desc and timestamp asc:")
ratings.orderBy(col("rating").desc(), col("timestamp").asc()).show(10)


#1-5
from pyspark.sql.functions import when, log10
ratings_with_derived = ratings.select(
    "*",
    (col("rating") * 2).alias("rating_x2"),
    when(col("rating") >= 4, 1).otherwise(0).alias("positive_rating"),
    log10(col("rating") + 1).alias("log_rating")
)
print("Ratings with derived columns:")
ratings_with_derived.show(10)

#1.6
from pyspark.sql.functions import lit
from pyspark.sql.types import StringType # Import StringType for explicit casting
ratings_with_nulls = ratings.withColumn("fake_null_col", lit(None).cast(StringType())) # Cast lit(None) to StringType
print("DataFrame with fake null column:")
ratings_with_nulls.show(5)
filled = ratings_with_nulls.fillna({"fake_null_col": "default_value"})
print("\nAfter filling nulls in fake_null_col:")
filled.show(5)
replaced = ratings.withColumn("modified_rating",
                            when(col("rating") == 5.0, 10.0).otherwise(col("rating")))
print("\nAfter replacing 5.0 with 10.0:")
replaced.filter(col("modified_rating") == 10.0).show(5)
dropped = ratings_with_nulls.dropna(subset=["fake_null_col"])
print(f"\nAfter dropping rows where fake_null_col is null: {dropped.count()} rows")


#1-7
# Count unique users
unique_users = ratings.select("userId").distinct().count()
print(f"Number of unique users: {unique_users}")
unique_movies_rated = ratings.select("movieId").distinct().count()
print(f"Number of unique movies rated: {unique_movies_rated}")
unique_movies_total = movies.select("movieId").distinct().count()
print(f"Number of unique movies in movies dataset: {unique_movies_total}")
duplicate_count = ratings.groupBy("userId", "movieId") \
                         .count() \
                         .filter(col("count") > 1) \
                         .count()
print(f"\nNumber of duplicate (userId, movieId) pairs: {duplicate_count}")
deduplicated = ratings.dropDuplicates(["userId", "movieId"])
print(f"Rows after deduplication: {deduplicated.count()} (original: {ratings.count()})")

Ratings DataFrame - First 10 rows:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
+------+-------+------+---------+
only showing top 10 rows

Movies DataFrame - First 10 rows:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|    

## Task 2 — Aggregations & GroupBy

**2.1 Simple aggregations**

*	Compute min, max, avg rating
*	Count total number of ratings
*	Count ratings per movieId

**2.2 GroupBy**

Compute:

*	average rating per movie
*	number of ratings per movie
*	average rating per user
*	number of ratings per user

**2.3 Top items**

*	Top 20 most-rated movies
*	Top 20 best-rated movies (min 50 ratings)

→ You must filter with a join or window

**2.4 Window functions**

Using Window partitioned by movieId:

*	rank users by timestamp (earliest → latest rating)
*	create a lag column: previous rating by same user
*	compute average rating per movie with window


In [ ]:
# Write your solution here

## Task 3 — Spark SQL

In [ ]:
ratings.createOrReplaceTempView("ratings_table")
movies.createOrReplaceTempView("movies_table")

**3.1 Simple SQL Queries**

*	Show first 20 rows
*	Count total ratings
*	Count distinct users
*	Average rating overall

**3.2 SQL Grouping**
*	Average rating by movie
*	Ratings count by movie
*	Best movies with at least 100 ratings
*	Worst movies with at least 100 ratings
*	Most active users

**3.3 SQL CASE WHEN**

Create rating buckets:

*	≥ 4.0 → “high”
*	3.0–3.9 → “medium”
*	< 3.0 → “low”

**3.4 SQL Window Functions**

*	Top 10 movies by rating for each genre
*	Earliest rating per user (row_number)
*	Rolling average rating per movie

In [ ]:
# Write your solution here

## Task 4 — Joins & Genre Analytics

**4.1 Basic join**

Join ratings → movies on movieId.

*	Show 20 joined rows
*	Show userId, title, rating
*	Count how many ratings each genre has

**4.2 Parse genres**

genres looks like "Action|Adventure|Sci-Fi"

*	split genres into an array
*	explode the genres
*	count ratings per genre
*	compute average rating per genre

**4.3 Join + Aggregation**

Compute:

*	average rating per genre
*	average rating per movie title
*	number of ratings per movie
*	number of ratings per genre per year (bonus: extract year from title)

**4.4 Left Anti Join**

Find:

*	movies in movies.csv with no ratings
*	number of such movies

In [ ]:
# Write your solution here

## Task 5 — MLlib Classification

**Goal: Build a binary classifier:**

Predict whether a user will give rating ≥ 4.0

5.1 Create label

Add a column:

In [ ]:
label = 1 if rating >= 4 else 0

**5.2 Feature engineering**

Create features:

	•	rating (as-is)
	•	timestamp
	•	normalized timestamp
	•	(optional) number of ratings by that user (using join or window)

Use VectorAssembler to pack them.

**5.3 Split data**

70% train, 30% test.

**5.4 Train model**

Train a Logistic Regression model.

	•	print coefficients
	•	print intercept
	•	print ROC AUC

**5.5 Evaluate**

Compute :

	•	accuracy
	•	precision
	•	recall
	•	confusion matrix (TP, FP, TN, FN)

**5.6 Task: Improve model**

try:

	•	Adding a log-transformed timestamp
	•	Using a DecisionTreeClassifier
	•	Using a RandomForestClassifier
	•	Comparing metrics

In [ ]:
# Write your solution here

## Task 6 — Performance & Execution

**6.1 Check partitions**

Get number of partitions for ratings and movies.

**6.2 Repartition and coalesce**

	•	explain difference
	•	show effect on shuffles
	•	check number of partitions with rdd.getNumPartitions()

**6.3 Cache**

Cache ratings:

	•	compute count
	•	compute average rating per movie twice
	•	measure difference using Python time

**6.4 explain(True)**

run explain() on:

	•	a join
	•	a groupBy
	•	a window function

And identify shuffle boundaries.

In [ ]:
# Write your solution here

## Stop Spark

In [ ]:
spark.stop()